### Import libraries

In [12]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import json

### Load dataset

In [13]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() in {"notebook", "notebooks"}:
    ROOT = CURRENT_DIR.parent
else:
    ROOT = CURRENT_DIR

PROCESSED_DIR = ROOT / "data" / "processed"
CLEAN_CSV_PATH = PROCESSED_DIR / "rice_leaf_images_clean.csv"

SPLIT_DIR = PROCESSED_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

if not CLEAN_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Clean dataset not found: {CLEAN_CSV_PATH}\n"
        "Please run 02_validate_and_explore.ipynb first."
    )

df = pd.read_csv(CLEAN_CSV_PATH)

### Validate Required Columns

In [14]:
required_columns = {"image_path", "label"}

if not required_columns.issubset(df.columns):
    raise ValueError(
        f"Dataset must contain columns: {required_columns}"
    )


### Check Whether Dataset Is Empty

In [15]:
if df.empty:
    raise ValueError("Clean dataset is empty.")

### Check Missing Values

In [16]:
if df["label"].isna().any() or df["image_path"].isna().any():
    raise ValueError("Dataset contains missing image paths or labels.")

### Dataset Information

In [17]:
print("Class Distribution:")
print(df["label"].value_counts().sort_index())

Class Distribution:
label
Healthy                  753
bacterial_leaf_blight    785
brown_spot               760
leaf_blast               521
leaf_scald               514
narrow_brown_spot        767
neck_blast               677
rice_hispa               770
sheath_blight            608
tungro                   556
Name: count, dtype: int64


### Check minimum images per category

In [18]:

class_counts = df["label"].value_counts()

display(class_counts.to_frame("image_count"))

if class_counts.min() < 7:
    raise ValueError(
        "Each category needs at least 7 images for this stratified split. "
        "Collect more images or adjust the split."
    )


,image_count
label,
bacterial_leaf_blight,785
rice_hispa,770
narrow_brown_spot,767
brown_spot,760
Healthy,753
neck_blast,677
sheath_blight,608
tungro,556
leaf_blast,521


### training, validation and testing sets

In [19]:

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


In [20]:
print("Training images:", len(train_df))
print("Validation images:", len(validation_df))
print("Testing images:", len(test_df))

print("Total:", len(train_df) + len(validation_df) + len(test_df))

Training images: 4697
Validation images: 1007
Testing images: 1007
Total: 6711


### Save CSV files

In [21]:

train_df.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

validation_df.to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

print("All three datasets saved successfully!")


All three datasets saved successfully!


### Create class mapping

In [22]:
class_names = sorted(df["label"].unique())

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

mapping_path = PROCESSED_DIR / "class_mapping.json"

with open(mapping_path, "w", encoding="utf-8") as file:
    json.dump(class_to_idx, file, indent=4)

print("Class mapping:")
print(class_to_idx)


Class mapping:
{'Healthy': 0, 'bacterial_leaf_blight': 1, 'brown_spot': 2, 'leaf_blast': 3, 'leaf_scald': 4, 'narrow_brown_spot': 5, 'neck_blast': 6, 'rice_hispa': 7, 'sheath_blight': 8, 'tungro': 9}
